# 06 Problem 2a: robustness to degradations
T4 GPU, about 30 min (CLIP features for every condition). No retraining: the 3 saved Problem 1 models are scored on degraded copies of the test sets.

Every condition is applied to real AND fake images (like a platform re-compressing every upload). Resize = down/up then back to 224, so all models still get the same input size.

Refs: Wang et al. 2020 (arXiv 1912.11035), Grommelt et al. 2024 (arXiv 2403.17608), UnivFD (arXiv 2302.10174)

In [ ]:
# setup (same in every notebook)
import os, sys, subprocess

try:  # autoreload breaks on some Colab Python versions
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload unavailable, restart runtime after pulling changes")

REPO_URL = "https://github.com/Devanshu1503/CV_project_genimage-detection.git"
REPO_DIR = "/content/CV_project_genimage-detection"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print("[setup] git pull:", (r.stdout or r.stderr).strip())
else:  # local run from notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.colab_setup import setup
cfg, P = setup()  # config dict + paths

## 1. GPU check

In [ ]:
import torch

ALLOW_CPU = False   # CPU is way too slow, debug only
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU OK:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    msg = ("NO GPU FOUND. In Colab: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart "
           "and run all again.")
    if not ALLOW_CPU:
        raise RuntimeError(msg)
    print("WARNING:", msg)

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from PIL import Image

from src import datasets as ds
from src import degrade as dg
from src import eda_utils as eu
from src import evaluate as ev
from src import models as mdl
from src import robustness as rb
from src.config import generator_names, get_logger

eu.set_style()
pd.set_option("display.width", 200, "display.max_columns", 30)
log = get_logger("06_problem2_degradation", P.logs)
SEED = cfg["project"]["seed"]
NAMES = generator_names(cfg)
GEN_ORDER = [g["folder"] for g in cfg["generators"]]
TRAIN_GEN = cfg["train_generator"]
THR = cfg["evaluation"]["threshold"]
FEAT = cfg["features"]
RCFG = cfg["training"]["resnet50"]
CONDS = cfg["problem2"]["conditions"]  # drop entries in config.yaml to save time
RES, FIG = P.results / "problem2", P.figures / "problem2"
PRED_DIR = RES / "preds"               # one csv per model x test set x condition, nb 08 reuses them
FEAT_ROOT = P.features / "problem2"
MODEL_ORDER = ["ResNet-50 (fine-tuned)", "CLIP ViT-L/14 + LR", "DINOv2 ViT-B/14 + LR"]
RESNET, CLIP, DINO = MODEL_ORDER


def save(fig, name):
    eu.savefig(fig, f"06_{name}", FIG)
    plt.show()


# fixed order so cached features line up
test = ds.load_index(P.manifests).query("split_set == 'test'").sort_values("sample_id").reset_index(drop=True)
print(f"{len(test)} test images, {len(CONDS)} conditions: {list(CONDS)}")

### Copy test images to local disk

In [ ]:
n = ds.stage_locally(test["processed_relpath"], P.project_root, P.work_dir)
ROOT = P.work_dir
print(f"Copied {n} new files")

## 2. What the degradations look like
96 px center crop, zoomed in so JPEG blocks and blur are visible.

In [ ]:
ex = pd.concat([test.loc[(test["generator"] == TRAIN_GEN) & (test["y"] == y)].iloc[[0]] for y in [1, 0]])
fig, axes = plt.subplots(2, len(CONDS), figsize=(1.35 * len(CONDS), 3.3))
for r, (_, row) in enumerate(ex.iterrows()):
    img = Image.open(ROOT / row["processed_relpath"]).convert("RGB")
    for c, (cond, steps) in enumerate(CONDS.items()):
        ax = axes[r, c]
        ax.imshow(dg.FixedDegrade(steps)(img, row["sample_id"]).crop((64, 64, 160, 160)))
        ax.axis("off")
        if r == 0:
            ax.set_title(cond, fontsize=8)
    axes[r, 0].text(-0.1, 0.5, "fake SD1.4" if row["y"] else "real", transform=axes[r, 0].transAxes,
                    rotation=90, ha="right", va="center", fontsize=9)
fig.tight_layout()
save(fig, "degradation_examples")

## 3. Score the 3 Problem 1 models on every condition
Resumable: finished (model, condition) csvs in `results/problem2/preds/` and feature shards are skipped.

In [ ]:
resnet = mdl.build_resnet50(weights=None)
ck = torch.load(P.checkpoints / "resnet50" / "best.pt", map_location="cpu", weights_only=False)
resnet.load_state_dict(ck["model"])
resnet = resnet.to(DEVICE).eval()
all_preds = [rb.score_resnet(resnet, test, ROOT, CONDS, RESNET, "genimage", PRED_DIR, RCFG["crop_size"],
                             RCFG["batch_size"], RCFG["num_workers"], DEVICE, RCFG["amp"], log)]

In [ ]:
# on Python 3.13 the DataLoader may print "can only test a child process", harmless
feat_kw = dict(image_size=cfg["preprocess"]["image_size"], batch_size=FEAT["batch_size"],
               shard_size=FEAT["shard_size"], num_workers=FEAT["num_workers"], device=DEVICE)
for kind, name in [("clip", CLIP), ("dinov2", DINO)]:
    clf = joblib.load(P.checkpoints / f"linear_probe_{kind}.joblib")
    lazy = rb.LazyExtractor(kind, cfg)
    all_preds.append(rb.score_probe(lazy, clf, test, ROOT, CONDS, name, "genimage", PRED_DIR, FEAT_ROOT, log, **feat_kw))
    lazy.free()
preds = pd.concat(all_preds, ignore_index=True)
preds.to_csv(RES / "degradation_predictions.csv", index=False)
print(preds.groupby(["model", "condition"], sort=False).size().unstack("model").head(3))

### Sanity check: clean condition = Problem 1 numbers

In [ ]:
res = rb.results_by_condition(preds, CONDS, GEN_ORDER, TRAIN_GEN, THR)
res.insert(3, "generator_name", res["generator"].map(NAMES).fillna(res["generator"]))
ev.save_table(res, RES / "degradation_results.csv")
p1 = pd.read_csv(P.results / "problem1_results.csv")
chk = res.loc[res["condition"] == "clean"].merge(p1[["model", "generator", "auroc"]], on=["model", "generator"],
                                                 suffixes=("", "_problem1"))
gap = (chk["auroc"] - chk["auroc_problem1"]).abs()
print(f"max |AUROC clean - AUROC problem 1| = {gap.max():.4f} over {len(chk)} rows (only fp16 noise expected)")

## 4. AUROC vs degradation level
`mean_all` = mean over all 8 test generators. Dashed line = chance.

In [ ]:
fig = rb.plot_degradation_curves(res, "mean_all", "auroc", MODEL_ORDER, title="Mean AUROC over all 8 generators")
save(fig, "auroc_vs_degradation_mean_all")
fig = rb.plot_degradation_curves(res, TRAIN_GEN, "auroc", MODEL_ORDER, title=f"AUROC on {NAMES[TRAIN_GEN]} (seen in training)")
save(fig, "auroc_vs_degradation_sd14")
fig = rb.plot_degradation_curves(res, "mean_unseen", "auroc", MODEL_ORDER, title="Mean AUROC over the 7 unseen generators")
save(fig, "auroc_vs_degradation_mean_unseen")

In [ ]:
cond_order = list(CONDS)
for g in ["mean_all", TRAIN_GEN, "mean_unseen"]:
    piv = (res.loc[res["generator"] == g].pivot(index="model", columns="condition", values="auroc")
              .reindex(index=MODEL_ORDER, columns=cond_order))
    print(f"\nAUROC, {NAMES.get(g, g)}")
    display(piv.style.format("{:.3f}").background_gradient(cmap="RdYlGn", vmin=0.5, vmax=1.0, axis=None))
    drop = piv.sub(piv["clean"], axis=0)
    print(f"Change vs clean, {NAMES.get(g, g)}")
    display(drop.style.format("{:+.3f}").background_gradient(cmap="RdBu", vmin=-0.4, vmax=0.4, axis=None))
    piv.to_csv(RES / f"degradation_auroc_{NAMES.get(g, g)}.csv")

### Per generator heatmaps

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(6 * len(MODEL_ORDER), 4), sharey=True)
for ax, m in zip(axes, MODEL_ORDER):
    piv = (res.loc[(res["model"] == m) & res["generator"].isin(GEN_ORDER)]
              .pivot(index="generator_name", columns="condition", values="auroc")
              .reindex(index=[NAMES[g] for g in GEN_ORDER], columns=cond_order))
    sns.heatmap(piv, ax=ax, vmin=0.3, vmax=1, cmap="RdYlGn", annot=True, fmt=".2f", annot_kws={"fontsize": 6}, cbar=False)
    ax.set(title=m, xlabel="", ylabel="")
    ax.tick_params(axis="x", rotation=90)
fig.tight_layout()
save(fig, "auroc_heatmap_generator_x_condition")

## 5. Who gets the errors: reals or fakes?
At the fixed 0.5 threshold. If fake accuracy collapses but real accuracy stays, degraded fakes are being called real.

In [ ]:
for metric in ["fake_acc", "real_acc"]:
    fig = rb.plot_degradation_curves(res, TRAIN_GEN, metric, MODEL_ORDER, title=f"{metric} on {NAMES[TRAIN_GEN]}")
    save(fig, f"{metric}_vs_degradation_sd14")

## Findings
Fill in after the run (numbers from the tables above).
- Clean sanity check: max gap vs Problem 1 = ...
- JPEG: how fast does each model drop (SD1.4 and mean_all) from clean to q50?
- Resize / blur / noise: which hurts most, and which model is most robust?
- Errors: under JPEG, does fake accuracy collapse (fakes called real) or real accuracy?
- Champion vs challenger: is CLIP still ahead of ResNet-50 once images are degraded?
- Motivation for nb 08: which conditions should augmentation fix?

## Run before closing the tab
Uploads pending Drive writes, otherwise the next session may not see them.

In [ ]:
from src.colab_setup import flush_drive
flush_drive()